# Modelo baseline para churn voluntário

Este notebook constrói um primeiro modelo interpretável para estimar risco de churn voluntário em clientes de um provedor de internet.

O objetivo do baseline não é produzir o melhor modelo possível. A função dele é criar uma referência simples, auditável e fácil de explicar antes de testar abordagens mais complexas.

Os dados usados aqui são sintéticos. As métricas e associações observadas não devem ser interpretadas como desempenho real de produção.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH_CANDIDATES = [
    Path("data/customers_churn_synthetic.csv"),
    Path("../data/customers_churn_synthetic.csv"),
]

DATA_PATH = next(path for path in DATA_PATH_CANDIDATES if path.exists())

df = pd.read_csv(DATA_PATH)

df.head()


,customer_id,tenure_months,access_technology,download_speed_mbps,monthly_fee,has_contract_loyalty,overdue_invoice_count,oldest_overdue_days,active_agreement_installment_amount,had_price_adjustment_90d,support_tickets_90d,repeat_issue_90d,avg_resolution_hours_90d,outage_count_30d,network_outage_hours_30d,churn_90d
0,CUST-000001,40,fiber,500,160.93,0,0,0,0.00,1,0,0,0.00,0,0.00,0
1,CUST-000002,53,fiber,200,109.99,1,0,0,0.00,0,0,0,0.00,0,0.00,0
2,CUST-000003,35,fiber,100,100.73,0,1,46,0.00,0,0,0,0.00,0,0.00,0
3,CUST-000004,31,fiber,200,116.91,1,0,0,61.27,0,0,0,0.00,1,1.13,1
4,CUST-000005,58,fiber,100,105.92,1,2,32,0.00,0,1,0,47.23,0,0.00,0


## Checagem inicial e definição do alvo

Antes de treinar qualquer modelo, vamos confirmar a estrutura da base e separar o alvo `churn_90d` das variáveis de entrada.

Essa separação evita incluir o próprio resultado que queremos prever como informação para o modelo.


In [2]:
print(f"Linhas: {df.shape[0]}")
print(f"Colunas: {df.shape[1]}")
print(f"Taxa de churn voluntário: {df['churn_90d'].mean():.2%}")

TARGET = "churn_90d"
FEATURES = [column for column in df.columns if column not in ["customer_id", TARGET]]

print(f"Variáveis de entrada: {len(FEATURES)}")
FEATURES


Linhas: 5000
Colunas: 16
Taxa de churn voluntário: 10.34%
Variáveis de entrada: 14


['tenure_months',
 'access_technology',
 'download_speed_mbps',
 'monthly_fee',
 'has_contract_loyalty',
 'overdue_invoice_count',
 'oldest_overdue_days',
 'active_agreement_installment_amount',
 'had_price_adjustment_90d',
 'support_tickets_90d',
 'repeat_issue_90d',
 'avg_resolution_hours_90d',
 'outage_count_30d',
 'network_outage_hours_30d']

## Separação entre treino e teste

A base será dividida em treino e teste.

O modelo aprende apenas com o conjunto de treino. O conjunto de teste fica separado para simular a avaliação em clientes ainda não vistos pelo modelo.

A separação usa estratificação para manter proporção parecida de churn nas duas bases.


In [3]:
from sklearn.model_selection import train_test_split

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

split_summary = pd.DataFrame(
    {
        "base": ["treino", "teste"],
        "linhas": [len(X_train), len(X_test)],
        "taxa_churn": [y_train.mean(), y_test.mean()],
    }
)
split_summary["taxa_churn"] = split_summary["taxa_churn"].mul(100).round(2)

split_summary


,base,linhas,taxa_churn
0,treino,3750,10.35
1,teste,1250,10.32


## Pré-processamento das variáveis

A regressão logística precisa receber apenas números. Por isso, vamos separar variáveis numéricas e categóricas.

As variáveis numéricas serão padronizadas para ficarem em escala comparável. A variável categórica `access_technology` será convertida em colunas indicadoras com one-hot encoding.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ["access_technology"]
numeric_features = [column for column in FEATURES if column not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

print(f"Variáveis numéricas: {len(numeric_features)}")
print(f"Variáveis categóricas: {len(categorical_features)}")

numeric_features, categorical_features


Variáveis numéricas: 13
Variáveis categóricas: 1


(['tenure_months',
  'download_speed_mbps',
  'monthly_fee',
  'has_contract_loyalty',
  'overdue_invoice_count',
  'oldest_overdue_days',
  'active_agreement_installment_amount',
  'had_price_adjustment_90d',
  'support_tickets_90d',
  'repeat_issue_90d',
  'avg_resolution_hours_90d',
  'outage_count_30d',
  'network_outage_hours_30d'],
 ['access_technology'])

## Modelos comparados

Vamos comparar três referências:

- `DummyClassifier`: prevê sempre a classe majoritária, ou seja, assume que ninguém vai cancelar.
- `LogisticRegression` sem `class_weight`: regressão logística padrão.
- `LogisticRegression` com `class_weight="balanced"`: modelo baseline atual, com maior atenção à classe minoritária.

O `predict` da regressão logística usa limiar padrão de `0,5`: clientes com score igual ou acima de `0,5` são classificados como churn previsto. Esse limiar ainda não foi otimizado para operação.


In [5]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

plain_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000)),
    ]
)

balanced_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ]
)

dummy_model = DummyClassifier(strategy="most_frequent")

plain_model.fit(X_train, y_train)
balanced_model.fit(X_train, y_train)
dummy_model.fit(X_train, y_train)

model = balanced_model

print("Modelos treinados com sucesso.")


Modelos treinados com sucesso.


## Comparação inicial das métricas

A acurácia isolada é inadequada neste problema porque o churn é minoritário. Um modelo ingênuo que prevê sempre “não churn” pode acertar muitos casos e ainda assim não encontrar nenhum cliente que cancelaria.

Além da acurácia, vamos olhar precision, recall, F1, ROC-AUC e Average Precision.


In [6]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate_classifier(name, fitted_model):
    y_pred = fitted_model.predict(X_test)
    y_score = fitted_model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "Average Precision": average_precision_score(y_test, y_score),
        "clientes_sinalizados": int(y_pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

model_comparison = pd.DataFrame(
    [
        evaluate_classifier("dummy_classe_majoritaria", dummy_model),
        evaluate_classifier("logistica_sem_class_weight", plain_model),
        evaluate_classifier("logistica_class_weight_balanced", balanced_model),
    ]
)

model_comparison[
    [
        "modelo",
        "accuracy",
        "precision",
        "recall",
        "f1",
        "roc_auc",
        "Average Precision",
        "clientes_sinalizados",
        "tn",
        "fp",
        "fn",
        "tp",
    ]
].round(4)


,modelo,accuracy,precision,recall,f1,roc_auc,Average Precision,clientes_sinalizados,tn,fp,fn,tp
0,dummy_classe_majoritaria,0.8968,0.0000,0.0000,0.0000,0.5000,0.1032,0,1121,0,129,0
1,logistica_sem_class_weight,0.8976,1.0000,0.0078,0.0154,0.6101,0.1780,1,1121,0,128,1
2,logistica_class_weight_balanced,0.6184,0.1345,0.4961,0.2116,0.6069,0.1808,476,709,412,65,64


## Leitura do `class_weight="balanced"`

O parâmetro `class_weight="balanced"` aumenta o peso dos casos de churn durante o treino. Isso faz o modelo prestar mais atenção à classe minoritária.

O efeito esperado é aumentar o recall, ou seja, encontrar mais cancelamentos potenciais. A contrapartida é aumentar falsos positivos, porque mais clientes passam a ser sinalizados como risco.

Por esse motivo, as saídas de `predict_proba` neste baseline devem ser usadas principalmente como score de ordenação. Elas não devem ser interpretadas diretamente como probabilidades calibradas de churn.


## Matriz de confusão da regressão logística balanceada

A matriz de confusão mostra os acertos e erros do modelo com limiar padrão de `0,5`.

Para o negócio, os pontos mais importantes são:

- verdadeiros positivos: clientes que seriam abordados e realmente cancelariam;
- falsos positivos: clientes que seriam abordados, mas não cancelariam;
- falsos negativos: clientes que cancelariam, mas não seriam identificados pelo modelo;
- verdadeiros negativos: clientes corretamente deixados fora da abordagem.


In [7]:
y_pred_balanced = balanced_model.predict(X_test)
y_score_balanced = balanced_model.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_balanced).ravel()

confusion_table = pd.DataFrame(
    [[tn, fp], [fn, tp]],
    index=["Real: não churn", "Real: churn"],
    columns=["Previsto: não churn", "Previsto: churn"],
)

business_summary = pd.DataFrame(
    {
        "indicador": [
            "clientes abordados",
            "cancelamentos encontrados",
            "alertas falsos",
            "cancelamentos não encontrados",
        ],
        "quantidade": [fp + tp, tp, fp, fn],
        "interpretação": [
            "clientes classificados como risco e que receberiam ação de retenção",
            "clientes com churn voluntário que o modelo conseguiu sinalizar",
            "clientes sinalizados como risco, mas que não cancelaram",
            "clientes que cancelaram, mas ficaram fora da priorização pelo limiar 0,5",
        ],
    }
)

display(confusion_table)
display(business_summary)


,Previsto: não churn,Previsto: churn
Real: não churn,709,412
Real: churn,65,64


,indicador,quantidade,interpretação
0,clientes abordados,476,clientes classificados como risco e que recebe...
1,cancelamentos encontrados,64,clientes com churn voluntário que o modelo con...
2,alertas falsos,412,"clientes sinalizados como risco, mas que não c..."
3,cancelamentos não encontrados,65,"clientes que cancelaram, mas ficaram fora da p..."


## Cenários operacionais por capacidade

Na prática, a área de retenção pode não conseguir abordar todos os clientes sinalizados. Por isso, também vamos avaliar cenários baseados em ranking.

Aqui usamos o score do modelo para ordenar os clientes do maior para o menor risco e simulamos três capacidades de abordagem:

- 10% dos clientes com maior score;
- 20% dos clientes com maior score;
- 30% dos clientes com maior score.

Esses percentuais são cenários de capacidade, não limiares otimizados no conjunto de teste.


In [8]:
def evaluate_capacity_scenario(y_true, scores, share):
    ranking = pd.DataFrame(
        {
            "churn_90d": y_true.to_numpy(),
            "score": scores,
        }
    ).sort_values("score", ascending=False)

    prioritized_count = int(np.ceil(len(ranking) * share))
    prioritized = ranking.head(prioritized_count)
    churns_found = int(prioritized["churn_90d"].sum())
    total_churns = int(ranking["churn_90d"].sum())
    precision = churns_found / prioritized_count
    recall = churns_found / total_churns
    overall_churn_rate = ranking["churn_90d"].mean()
    lift = precision / overall_churn_rate

    return {
        "cenário": f"top {int(share * 100)}%",
        "clientes_priorizados": prioritized_count,
        "churns_encontrados": churns_found,
        "precision": precision,
        "recall": recall,
        "lift_vs_taxa_geral": lift,
    }

capacity_scenarios = pd.DataFrame(
    [evaluate_capacity_scenario(y_test, y_score_balanced, share) for share in [0.10, 0.20, 0.30]]
)

capacity_scenarios.round(4)


,cenário,clientes_priorizados,churns_encontrados,precision,recall,lift_vs_taxa_geral
0,top 10%,125,24,0.1920,0.1860,1.8605
1,top 20%,250,38,0.1520,0.2946,1.4729
2,top 30%,375,52,0.1387,0.4031,1.3437


## Coeficientes do modelo

A regressão logística permite inspecionar os coeficientes aprendidos após o pré-processamento.

Coeficientes positivos estão associados a maior score de churn no dataset sintético. Coeficientes negativos estão associados a menor score.

Essa leitura é associativa, não causal. Os coeficientes mostram padrões aprendidos a partir das relações simuladas no dataset, mas não provam que uma variável causa cancelamento.


In [9]:
feature_names = balanced_model.named_steps["preprocessor"].get_feature_names_out()
feature_names = [
    name.replace("numeric__", "").replace("categorical__", "")
    for name in feature_names
]

coefficients = pd.Series(
    balanced_model.named_steps["classifier"].coef_[0],
    index=feature_names,
).sort_values(ascending=False)

positive_coefficients = coefficients.head(6).reset_index()
positive_coefficients.columns = ["variável", "coeficiente"]

negative_coefficients = coefficients.tail(6).sort_values().reset_index()
negative_coefficients.columns = ["variável", "coeficiente"]

print("Maiores coeficientes positivos")
display(positive_coefficients.round(4))

print("Maiores coeficientes negativos")
display(negative_coefficients.round(4))


Maiores coeficientes positivos


,variável,coeficiente
0,oldest_overdue_days,0.2055
1,download_speed_mbps,0.1808
2,support_tickets_90d,0.1721
3,had_price_adjustment_90d,0.1470
4,repeat_issue_90d,0.1040
5,avg_resolution_hours_90d,0.0885


Maiores coeficientes negativos


,variável,coeficiente
0,has_contract_loyalty,-0.1904
1,monthly_fee,-0.1830
2,active_agreement_installment_amount,-0.0684
3,access_technology_radio,-0.0625
4,overdue_invoice_count,-0.0097
5,tenure_months,0.0179


## Conclusões do baseline

O que o modelo aprendeu:

- O baseline encontrou sinais associados ao churn voluntário no dataset sintético.
- O uso de `class_weight="balanced"` aumentou a capacidade de encontrar clientes churn, mas também elevou a quantidade de falsos positivos.
- A acurácia sozinha não serve como métrica principal, porque modelos que ignoram o churn podem parecer bons apenas por causa do desbalanceamento da base.
- O ranking por score é mais útil para operação do que interpretar a saída como uma decisão automática.

Limitações:

- Os dados são sintéticos e refletem hipóteses, não comportamento real validado em produção.
- As probabilidades não foram calibradas.
- O limiar `0,5` não foi escolhido com base em custo operacional, capacidade de atendimento ou valor econômico.
- As associações dos coeficientes não devem ser interpretadas como causalidade.

Por que ainda não está pronto para produção:

- Falta validação com dados reais e janelas temporais bem definidas.
- Falta avaliar estabilidade temporal, custo de falso positivo, custo de falso negativo e impacto operacional.
- Falta calibrar probabilidades caso elas sejam usadas em comunicação de risco.
- Falta governança para uso de dados, monitoramento e revisão periódica.

Função do próximo modelo:

- O próximo modelo deve servir como comparação contra este baseline interpretável.
- Ele poderá capturar relações não lineares e interações mais complexas.
- A comparação deve verificar se o ganho de desempenho compensa a perda de simplicidade e explicabilidade.
